# Gumbel `c_scale` A/B — strengthen the prior anchor

Build 1 shipped (+9.4% vs control on every paired seed). The win came from
**anchoring selection/distillation to the policy prior**, and `gumbel_c_scale`
dials that anchor: **lower = stronger anchoring**. This A/B tests
`c_scale ∈ {0.5, 0.25}` against the shipped `c_scale=1.0` arm
(`v2_exit_gumbel_on_a100`) and the no-gumbel control (`v2_exit_gumbel_ctrl_a100`).

**Each session trains ONE arm** — set `GUMBEL_C_SCALE` below and run; start a
second Colab session with the other value to parallelize. Each arm gets its own
`run_name` (`v2_exit_gumbel_cs050_a100` / `cs025`) so Drive never collides.

Warm-starts from `v2_exit_a100/ckpt_000020.pt` on Drive (BC skipped). Same
A100 overrides as the on/ctrl arms in `train_colab.ipynb`, so the comparison is
apples-to-apples. GPU runtime recommended (any GPU; the net is tiny).


In [ ]:
#@title 1. Setup: Mount Drive, Clone Repo, Install Dependencies

import os, sys

from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/orbit_wars_outputs'
for sub in ['', '/checkpoints', '/logs', '/submissions']:
    os.makedirs(DRIVE_OUTPUT + sub, exist_ok=True)

REPO_URL = 'https://github.com/oshbocker/orbit_wars.git'  # <-- EDIT if needed
REPO_DIR = '/content/orbit_wars'
if os.path.exists(REPO_DIR):
    print('Repo present, pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

os.system('pip install -q --upgrade "kaggle-environments>=1.28.0" pyyaml tensorboard')

CPU_COUNT = os.cpu_count() or 8

# Silence kaggle_environments' noisy OpenSpiel INFO logs in THIS kernel
# (subprocess training output is filtered by grep on the launch line instead).
import logging
for _ln in ('kaggle_environments.envs.open_spiel_env.open_spiel_env',
            'kaggle_environments.core_harness'):
    logging.getLogger(_ln).disabled = True

print(f'\nDrive: {DRIVE_OUTPUT} | cwd: {os.getcwd()} | vCPUs: {CPU_COUNT}')


In [ ]:
#@title 2. Train one c_scale arm (run 0.5 and 0.25 in separate sessions)
import os, yaml
from v2.config import load_v2_config, v2_config_to_dict

GUMBEL_C_SCALE = 0.5  #@param [0.5, 0.25] {type:"raw"}

cfg = load_v2_config('configs/v2_exit_gumbel.yaml')
cfg.device = 'cuda'
cfg.save_dir = f'{DRIVE_OUTPUT}/checkpoints'
cfg.log_dir = f'{DRIVE_OUTPUT}/logs'

# Same A100 overrides as the gumbel on/ctrl arms in train_colab.ipynb.
cfg.exit.iterations = 60
cfg.exit.games_per_iter = 12
cfg.exit.search_depth = 16
cfg.exit.search_candidates = 12
cfg.exit.train_epochs = 6
cfg.exit.train_batch_size = 2048
cfg.exit.dataset_max_iters = 4
cfg.exit.search_workers = CPU_COUNT
cfg.exit.collect_workers = CPU_COUNT
cfg.exit.collect_fast_env = True
cfg.checkpoint_every = 5
cfg.eval.eval_every = 5
cfg.eval.eval_games = 40
cfg.eval.eval_seed = 20000
cfg.eval.eval_workers = CPU_COUNT

# The experiment: Build 1 with a stronger prior anchor.
cfg.exit.gumbel_search = True
cfg.exit.net_opponent = False
cfg.exit.gumbel_c_scale = float(GUMBEL_C_SCALE)
cfg.run_name = f'v2_exit_gumbel_cs{int(round(float(GUMBEL_C_SCALE) * 100)):03d}_a100'

WARM_START = f'{DRIVE_OUTPUT}/checkpoints/v2_exit_a100/ckpt_000020.pt'
CHECKPOINT_DIR = f'{cfg.save_dir}/{cfg.run_name}'
resume_ckpt = f'{CHECKPOINT_DIR}/ckpt_last.pt'
if os.path.exists(resume_ckpt):
    resume_flag = f'--resume "{resume_ckpt}"'      # continue this arm
elif os.path.exists(WARM_START):
    resume_flag = f'--resume "{WARM_START}"'       # warm-start (skips BC)
else:
    raise FileNotFoundError(
        f'No warm-start checkpoint at {WARM_START} — imitation is off in this '
        'config, so training would start from RANDOM weights. Check Drive.')

temp_cfg = '/tmp/v2_cscale_cfg.yaml'
with open(temp_cfg, 'w') as f:
    yaml.dump(v2_config_to_dict(cfg), f, sort_keys=True)
print(f'run_name={cfg.run_name}  c_scale={cfg.exit.gumbel_c_scale}')
print(f'resume: {resume_flag}\n')

!python -m v2.exit_train --config {temp_cfg} {resume_flag} 2>&1 | grep -vE --line-buffered "^\[kaggle_environments\.(envs\.open_spiel_env|core_harness)"

print(f'\nCheckpoints saved to: {CHECKPOINT_DIR}')


In [ ]:
#@title 3. Gate: c_scale arms vs control AND the shipped c_scale=1.0 arm
from pathlib import Path
from v2.config import load_v2_config, v2_config_to_dict
from scripts.eval_fast import _eval_checkpoint

CONTROL_RUN  = 'v2_exit_gumbel_ctrl_a100'
VARIANT_RUNS = ['v2_exit_gumbel_on_a100',      # shipped c_scale=1.0 reference
                'v2_exit_gumbel_cs050_a100',
                'v2_exit_gumbel_cs025_a100']
ITER         = 'last'                  #@param {type:"string"}  # 'last' or e.g. '35'
GAMES        = 60                      #@param {type:"integer"} # per seed batch
SEED_BATCHES = [20000, 31000, 42000]
EVAL_WORKERS = CPU_COUNT

cfg_dict = v2_config_to_dict(load_v2_config('configs/v2_exit_gumbel.yaml'))
ckpt_root = Path(f'{DRIVE_OUTPUT}/checkpoints')
ckpt_name = 'ckpt_last.pt' if ITER == 'last' else f'ckpt_{int(ITER):06d}.pt'

def score(run):
    path = ckpt_root / run / ckpt_name
    if not path.exists():
        return None
    wrs = []
    for s in SEED_BATCHES:
        win, loss, tie = _eval_checkpoint(path, cfg_dict, GAMES, EVAL_WORKERS, 'apex', s)
        wrs.append(win / max(win + loss + tie, 1))
    return wrs

print(f'iter={ITER}  games/seed={GAMES}  seeds={SEED_BATCHES}  workers={EVAL_WORKERS}\n')
header = f'{"run":>30} | ' + ' '.join(f'seed{s:>6}' for s in SEED_BATCHES) + f' | {"mean":>5}'
print(header); print('-' * len(header))
results = {}
for run in [CONTROL_RUN] + VARIANT_RUNS:
    wrs = score(run)
    if wrs is None:
        print(f'{run:>30} | (no checkpoint at {run}/{ckpt_name} — train it first)')
        continue
    results[run] = wrs
    print(f'{run:>30} | ' + ' '.join(f'{r:>9.0%}' for r in wrs) + f' | {sum(wrs)/len(wrs):>5.0%}')

if CONTROL_RUN in results:
    ctrl = results[CONTROL_RUN]
    print('\nvs control, per-seed win-rate delta:')
    for run, wrs in results.items():
        if run == CONTROL_RUN: continue
        d = [a - b for a, b in zip(wrs, ctrl)]
        print(f'  {run}: ' + ' '.join(f'{x:+.0%}' for x in d)
              + f'   mean {sum(d)/len(d):+.1%}   '
              + ('beats control on every seed' if all(x > 0 for x in d) else 'not on every seed'))
    if 'v2_exit_gumbel_on_a100' in results:
        ref = results['v2_exit_gumbel_on_a100']
        print('\nvs the shipped c_scale=1.0 arm (the bar a c_scale variant must clear):')
        for run, wrs in results.items():
            if run in (CONTROL_RUN, 'v2_exit_gumbel_on_a100'): continue
            d = [a - b for a, b in zip(wrs, ref)]
            print(f'  {run}: ' + ' '.join(f'{x:+.0%}' for x in d)
                  + f'   mean {sum(d)/len(d):+.1%}   '
                  + ('SHIP over c_scale=1.0 (every seed)' if all(x > 0 for x in d) else 'not on every seed'))
